In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array osserations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "AreaAverages_ColdPools"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf)

    else:  # 2D variable case
        shape = (ModelData.Ntime, 1)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output


def GetMean(variableSubset):
    variableMean = variableSubset.mean(dim=("latitude","longitude"), skipna=True).data
    return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def LoadAdditionalVariableData(ModelData, variableName,t,printout=True):

    codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "AdditionalVariables")
    dataType = "DensityPotentialTemperature"
    outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

    outputFolder = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}_spinup{ModelData.spinup_hours}hrs"
    outputFolderPath = os.path.join(outputDirectory,outputFolder,variableName)
    os.makedirs(outputFolderPath, exist_ok=True)
    
    outputFile = f"{variableName}_{ModelData.timeStrings[t]}.nc"
    outputFilePath = os.path.join(outputFolderPath,outputFile)
        
    outputData = xr.open_dataset(outputFilePath)

    if printout == True:
        print(f"Loaded from {outputFilePath}","\n")

    outputData = outputData["__xarray_dataarray_variable__"]
    return outputData
    
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def MeanDBZ(variableSubset):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
def ComputeOutflowVelocity(uField, vField, thetaRhoField):
    """
    Computes Eulerian outflow velocity on a latitude/longitude grid.
    uField, vField must be in m/s.
    thetaRhoField must be in K.

    Physical gradient on a spherical lat/lon grid:

        d(theta)/dx = (1 / (R * cos(phi))) * d(theta)/d(lambda)
        d(theta)/dy = (1 / R) * d(theta)/d(phi)

    where:
        lambda = longitude (radians)
        phi    = latitude  (radians)
        R      = Earth radius

    Outward unit vector:

        nx = (d(theta)/dx) / |grad(theta)|
        ny = (d(theta)/dy) / |grad(theta)|

    Outflow velocity:

        Uout = u * nx + v * ny
    """

    import numpy as np

    # Earth's radius (meters)
    R = 6_371_000  

    latRad = np.deg2rad(thetaRhoField.latitude)

    # --- compute correct physical gradients ---
    dthdlat = thetaRhoField.differentiate("latitude")    # K per degree latitude
    dthdlon = thetaRhoField.differentiate("longitude")   # K per degree longitude

    # Convert degrees → meters
    dthdy = dthdlat * (np.pi/180) * R
    dthdx = dthdlon * (np.pi/180) * R * np.cos(latRad)

    # --- gradient magnitude ---
    gradMag = np.sqrt(dthdx**2 + dthdy**2)

    # --- unit normal vector ---
    nx = dthdx / gradMag
    ny = dthdy / gradMag

    # --- project wind onto outward direction ---
    Uout = uField * nx + vField * ny
    Uout.name = "outflow_velocity"

    return Uout

In [ ]:
def RunCalculations(varNames, 
                    loop_elements):
    outputDictionary={}
    
    for count, t in enumerate(tqdm(range(loop_elements), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")

        #Loading Data
        [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        #getting threshold variables
        th_rho = LoadAdditionalVariableData(ModelData, variableName="Theta_rho", t=t)
        th_rho_prime = th_rho - th_rho.mean()
        qr = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName="qr").isel(nVertLevels=0)
        

        
        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            #Loading Data
            if varName == "Theta_rho":
                variableSubset = th_rho
            elif varName == "Theta_rho_prime":
                variableSubset = th_rho - th_rho.mean()
            elif "qv" in varName:
                variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName="qv").isel(nVertLevels=0)
                if varName == "qv_prime":
                    variableSubset -= variableSubset.mean()
            elif varName in ['w']:
                variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName).isel(nVertLevelsP1=0)
            elif varName in ['VEL','OUTFLOW']:
                u= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName="uReconstructZonal").isel(nVertLevels=0)
                v= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName="uReconstructMeridional").isel(nVertLevels=0)
                if varName == "VEL":
                    variableSubset = np.sqrt(u**2+v**2)
                elif varName == "OUTFLOW":
                    variableSubset = ComputeOutflowVelocity(u,v,th_rho)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)

            #Applying ColdPools Threshold    
            cond1 = th_rho_prime < -1
            cond2 = qr > 1e-6
            variableSubset = variableSubset.where(cond1&cond2)
            
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset, fill_nan=False)
                outputDictionary[varName] = output

            #Taking Mean
            if varName in ['refl10cm','refl10cm_1km']:
                variableMean = MeanDBZ(variableSubset)  
            else:
                variableMean = GetMean(variableSubset)
                
            outputDictionary[varName][t] = variableMean
    return outputDictionary

In [ ]:
def GetData_Subset(ModelData,t):  
    data = ModelData.GetDataTimestep(t,printout=False)
    data_diag = ModelData.GetDataTimestep_diag(t,printout=False)
    
    [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
    [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
    dataSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(data,latBounds, lonBounds)
    dataSubset_diag, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(data_diag,latBounds, lonBounds)
    dataSubset_static, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(ModelData.staticData,latBounds, lonBounds)

    # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
    return dataSubset, dataSubset_diag, dataSubset_static, lat, lon, data, data_diag

In [ ]:
def RunAreaAverages(ModelData,varNames,name, 
                    loop_elements):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}_{loop_elements[0]}-{loop_elements[-1]+1}.h5")
    
    #loading back in 
    try:
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(varNames, 
                                           loop_elements) #takes about 10 minutes
        #saving output
        
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
#running
def GetDictionary_1(ModelData):
    #2D Variables (12 vars)
    #surface variables
    varNames = ["Theta_rho","Theta_rho_prime","qv","qv_prime","w","VEL","OUTFLOW"]
    outputDictionary_1 = RunAreaAverages(ModelData,varNames, "1",
                                         loop_elements)
    return outputDictionary_1

In [ ]:
def RunJob(loop_elements):
    #getting NSSL dictionaries
    RunType = (Region,Case,"NSSL",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
    
    outputDictionary_2D_NSSL = GetDictionary_1(ModelData,
                                               loop_elements)
    
    #getting TEMPO dictionaries
    RunType = (Region,Case,"TEMPO",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
    
    outputDictionary_2D_TEMPO = GetDictionary_1(ModelData,
                                                loop_elements)
    return outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO

In [ ]:
####################################
#CALCULATING
running = True #keep true when job_array is running
# running = False

In [ ]:
if running:
    [outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO] = RunJob(loop_elements)

In [ ]:
####################################
#RECOMBINING
recombining = False #keep false when job_array is running
# recombining = True

In [ ]:
def AddDictionaries(dictA, dictB):
    """
    Modifies dictA by adding dictB values into it
    """
    for key in dictA:
        dictA[key] += dictB[key]
        
def Recombine():
    for job_id in tqdm(range(1, num_jobs + 1)):
    
        start_job, end_job = JobArray._get_job_range(job_id)
        loop_elements = GetLoopElements(start_job, end_job)
    
        dict2D_NSSL, dict2D_TEMPO = RunJob(loop_elements)
    
        if job_id == 1:
            dict2D_NSSL_all = dict2D_NSSL
            dict2D_TEMPO_all = dict2D_TEMPO
        else:
            AddDictionaries(dict2D_NSSL_all,  dict2D_NSSL)L)
            AddDictionaries(dict2D_TEMPO_all, dict2D_TEMPO)
    return dict2D_NSSL_all, dict2D_TEMPO_all


In [ ]:
if recombining:
    [outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO] = Recombine()

In [ ]:
####################################
#PLOTTING FUNCTIONS
plotting = False #keep false when job array is running
# plotting = True

In [ ]:
def GetVerticalCoord(dataSubset):
    pressure_profile = dataSubset['pressure'].mean(dim=("latitude","longitude")).data
    dp = pressure_profile[-1] - pressure_profile[-2]
    p_topface = pressure_profile[-1] + dp  # extrapolate linearly
    pressure_profile_face = np.append(pressure_profile, p_topface)
    return (pressure_profile/100,pressure_profile_face/100)

if plotting:
    [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t=0)
    pressure_profiles = GetVerticalCoord(dataSubset)
    time_strings = ModelData.timeStrings
    time = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
#Helper Functions

def nansubtract(a, b):
    """
    Element-wise subtraction (a - b) that preserves NaNs.

    If shapes differ, raises a ValueError.
    """
    if a.shape != b.shape:
        raise ValueError(f"Shape mismatch: a{a.shape} != b{b.shape}")

    return np.where(np.isnan(a) | np.isnan(b), np.nan, a - b)

# Example: align datetime x-limits to min/max of your data
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

def AlignAxesRight(ax_list):
    """
    Aligns the right edges of all axes in ax_list (e.g., contour + line plots),
    so that colorbars don't make some axes narrower.

    It uses the first axis that contains a contour or image
    (typically a contourf plot) as the reference width.
    """

    # Try to find a contour axis (has .collections or .images)
    ref_ax = None
    for ax in ax_list:
        if getattr(ax, "collections", []) or getattr(ax, "images", []):
            ref_ax = ax
            break

    # If no contour axis found, just use the first axis
    if ref_ax is None:
        ref_ax = ax_list[0]

    ref_pos = ref_ax.get_position()

    # Apply its width to all other axes
    for ax in ax_list:
        pos = ax.get_position()
        new_pos = [pos.x0, pos.y0, ref_pos.width, pos.height]
        ax.set_position(new_pos)

    print(f"Aligned {len(ax_list)} axes using reference width from contour axis at {ref_pos.width:.3f}")
# #EXAMPLE USAGE
# fig, axs = plt.subplots(2, 1, figsize=(8, 6))

# # contourf on top, line on bottom
# time = np.arange(24)
# pressure = np.linspace(1000, 100, 25)
# data = np.sin(time / 3)[None, :] * np.exp(-pressure[:, None] / 1000)

# plot = axs[0].contourf(time, pressure, data, cmap="RdBu_r")
# plt.colorbar(plot, ax=axs[0], orientation="vertical", pad=0.02)
# axs[1].plot(time, np.sin(time / 3), color="k")

# # Align both
# AlignAxesRight(axs)

# plt.show()

from matplotlib.ticker import MultipleLocator
def add_minor_white_grid(ax, alpha=0.5, lw=1.0, thickness=1.4, color='lightgray'):
    """
    Add white semi-transparent grid lines halfway between major ticks
    on both x and y axes (for contour plots).
    """
    from matplotlib.ticker import MultipleLocator

    # --- Minor locators at half the major spacing ---
    try:
        major_x = ax.xaxis.get_major_locator()
        step_x = major_x()[1] - major_x()[0]
        ax.xaxis.set_minor_locator(MultipleLocator(step_x / 2))
    except Exception:
        pass

    try:
        major_y = ax.yaxis.get_major_locator()
        step_y = major_y()[1] - major_y()[0]
        ax.yaxis.set_minor_locator(MultipleLocator(step_y / 2))
    except Exception:
        pass

    # --- Grid styling ---
    ax.grid(True, which="major", color=color, alpha=alpha, lw=lw * thickness)
    ax.grid(True, which="minor", color=color, alpha=alpha, lw=lw)

def AdjustLayout(fig,
                 left=0.07, right=0.97, bottom=0.07,
                 wspace=0.35, hspace=0.6,
                 title_space_inches=0.9, 
                 title_y_inches_from_top=0.25):
    """
    Applies a robust manual Matplotlib layout
    to a figure, reserving absolute space for a suptitle.
    """
    
    # Get figure height in inches
    fig_height_inches = fig.get_figheight()
    
    # Calculate the 'top' margin (where plots end) in relative figure coords
    # This leaves 'title_space_inches' at the top.
    top_margin = 1.0 - (title_space_inches / fig_height_inches)
    
    # Calculate the 'y' position for the suptitle
    title_y_relative = 1.0 - (title_y_inches_from_top / fig_height_inches)
    
    # Apply the manual layout
    plt.subplots_adjust(left=left, right=right, bottom=bottom, 
                        top=top_margin, wspace=wspace, hspace=hspace)

    # Return the calculated 'y' coordinate for the suptitle
    return title_y_relative

In [ ]:
def SaveFigure(fig, combinedDict, key,label_text):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{label_text}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"AreaAverages_{key}.png"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
def PlotSingle(axis, outputDictionarys, varName, time, pressure_profiles, labels,
               plottype="TZ", clim=None,num_levels=21):
    """
    Plot one variable on a given Matplotlib axis.
    Supports either:
      - A single dictionary (for single-model plots)
      - Two dictionaries (for model comparisons or line overlays)
    clim: tuple (vmin, vmax) for consistent color scaling (ignored for reflectivity)
    """

    # ------------------------------------------------------
    #  Helper: Line Plot
    # ------------------------------------------------------
    def lineplot(time, output, varName, units, color, label):
        axis.plot(time, output.squeeze(), color=color, label=label)
        axis.set_ylabel(f"{varName} " + fr"$({units})$")
        axis.set_xlabel("Time")
        axis.grid(True)
        SetXLimitsDatetime(axis, time)

    # ------------------------------------------------------
    #  Helper: Consistent Colorbar Formatting
    # ------------------------------------------------------
    def add_colorbar(fig, mappable, ax, label, ticks=None, orientation="vertical"):
        """Add a consistently styled, larger colorbar."""
        cbar = fig.colorbar(
            mappable, ax=ax, orientation=orientation,
            fraction=0.12, pad=0.020, aspect=20, shrink=1.15
        )
        cbar.set_label(label, fontsize=11)
        cbar.ax.tick_params(labelsize=8, width=1.1, length=4, pad=2)
        if ticks is not None:
            cbar.set_ticks(ticks)
        # Prevent overcrowding
        if len(cbar.get_ticks()) > 10:
            from matplotlib.ticker import MaxNLocator
            cbar.ax.yaxis.set_major_locator(MaxNLocator(8))
        return cbar

    # ------------------------------------------------------
    #  Units and scaling
    # ------------------------------------------------------
    if varName not in ["Theta_rho"]:
        if varName in ["VEL","OUTFLOW"]:
            units = "m/s"
        elif varName in ["Theta_rho_prime"]:
            units = "K"
        elif varName in ["qv_prime"]:
            units = "kg/kg"
        else:
            units = ModelData.GetUnits_Specific(varName).replace(" ", r"\ ")
        
    else: 
        units = "K"
    axisTitle = varName.replace("divergence", "convergence")
    if varName in ["qv", "qc", "qi", "qr", "qg", "q2", "qfx","qv_prime"]:
        multiplier = 1e3
        units = units.replace('kg', 'g', 1)
    elif varName == "divergence":
        multiplier = -1
    else:
        multiplier = 1

    # ------------------------------------------------------
    #  Select pressure profile
    # ------------------------------------------------------
    # sample_dict = outputDictionarys[0]
    # output_sample = sample_dict[varName]
    # pressure_profile = (
    #     pressure_profiles[0]
    #     if output_sample.shape[1] == pressure_profiles[0].shape[0]
    #     else pressure_profiles[1]
    # )
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    if varName not in ['w']:
        zlevels_plot = 0.5 * (zlevels[:-1] + zlevels[1:])
    else:
        zlevels_plot = zlevels.copy()

    # ------------------------------------------------------
    #  Case 1: Single-model plotting
    # ------------------------------------------------------
    if len(outputDictionarys) == 1:
        output = multiplier * outputDictionarys[0][varName]

        # Choose color setup
        if varName in ["w", "divergence"]:
            cmap = "RdBu_r"
        elif varName in ["refl10cm", "refl10cm_1km"]:
            cmap = None  # handled separately
        else:
            cmap = "viridis"

        # --- Line plot ---
        if output.ndim == 1 or output.shape[1] == 1:
            color = "k"
            label = labels[0] if labels else None
            lineplot(time, output, axisTitle, units, color, label)

        # --- Contour plot ---
        else:
            if plottype == "TZ" and varName not in ["refl10cm", "refl10cm_1km"]:
                # Apply shared clim via levels
                if clim is not None:
                    c0 = multiplier * clim[0]
                    c1 = multiplier * clim[1]
                    cmin, cmax = sorted([c0, c1])
                    levels = np.linspace(cmin, cmax, num_levels)
                    # levels = multiplier*np.linspace(clim[0], clim[1], num_levels)
                else:
                    levels = num_levels

                # Symmetric norm for diverging fields
                if varName in ["w", "divergence"]:
                    norm = TwoSlopeNorm(vcenter=0.0, vmin=clim[0] if clim else np.nanmin(output),
                                        vmax=clim[1] if clim else np.nanmax(output))
                else:
                    norm = None

                plot = axis.contourf(time, zlevels_plot, output.T, cmap=cmap,
                                     levels=levels, norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis,
                             label=f"{axisTitle} " + fr"$({units})$")
                add_minor_white_grid(axis)
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel("Time")
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype == "TZ" and varName in ["refl10cm", "refl10cm_1km"]:
                cmap, norm, levels, ticks = RadarPlotting_Class.GetReflectivityColormap()
                plot = axis.contourf(time, zlevels_plot, output.T,
                                     levels=levels, cmap=cmap, norm=norm, extend='both')
                cbar = add_colorbar(axis.figure, plot, axis,
                                    label="Reflectivity (dBZ)", ticks=ticks)
                add_minor_white_grid(axis)
                RadarPlotting_Class.FormatReflectivityColorbar(
                    cbar, ticks, orientation='vertical', show_labels=False
                )
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel("Time")
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype == "T":
                mean_output = np.nanmean(output, axis=1)
                color = "k"
                label = labels[0] if labels else None
                lineplot(time, mean_output, axisTitle, units, color, label)

    # ------------------------------------------------------
    #  Case 2: Two-model plotting
    # ------------------------------------------------------
    else:
        output1 = multiplier * outputDictionarys[0][varName]
        output2 = multiplier * outputDictionarys[1][varName]
        label1, label2 = labels

        is_line = (
            output1.ndim == 1 and output2.ndim == 1
            or output1.shape[1] == 1 and output2.shape[1] == 1
        )

        if is_line:
            with np.errstate(invalid="ignore"):
                mean1 = np.nanmean(output1, axis=1) if output1.ndim > 1 else output1
                mean2 = np.nanmean(output2, axis=1) if output2.ndim > 1 else output2
            lineplot(time, mean1, axisTitle, units, "blue", label1)
            lineplot(time, mean2, axisTitle, units, "green", label2)
            axis.legend(loc="upper left")

        else:
            if plottype == "TZ":
                diff = nansubtract(output1, output2)
                cmap = plt.get_cmap("RdBu_r").copy()
                cmap.set_bad("black")
                axis.set_facecolor('black')
                vlim = np.nanmax(np.abs(diff))
                norm = TwoSlopeNorm(vcenter=0.0, vmin=-vlim, vmax=vlim)
                plot = axis.contourf(time, zlevels_plot, diff.T, cmap=cmap,
                                     levels=np.linspace(-vlim, vlim, 40),
                                     norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis,
                             label=f"Δ{axisTitle} " + fr"$({units})$")
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel("Time")
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype == "T":
                with np.errstate(invalid="ignore"):
                    mean1 = np.nanmean(output1, axis=1)
                    mean2 = np.nanmean(output2, axis=1)
                lineplot(time, mean1, axisTitle, units, "blue", label1)
                lineplot(time, mean2, axisTitle, units, "green", label2)
                axis.legend(loc="upper left")
                
                
    if varName in ["qr","qg","qc+qi","w"]:
        if plottype=="T":
            apply_scientific_notation([axis],dim='y')
        elif plottype=="TZ":
            apply_scientific_notation_colorbar([cbar])
    
    # ------------------------------------------------------
    #  Title and finish
    # ------------------------------------------------------
    axis.set_title(axisTitle, fontsize=11)

In [ ]:
def MakeCombinedPlot(combinedDict, key, plottype):
    """
    Creates a combined plot from a model-comparison dictionary:
    combinedDict = {"NSSL": {...}, "TEMPO": {...}, ...}

    For two models and TZ plots:
      Each row = variable
      Columns = [Model1, Model2, Difference]
    For T plots:
      Both models are overlaid on the same axes with distinct colors.
    """

    # --- Extract model and variable structure ---
    combinedDict2 = combinedDict[key]
    modelLabels = list(combinedDict2.keys())  # e.g. ["NSSL", "TEMPO"]
    first_model = modelLabels[0]
    varNames = list(combinedDict2[first_model].keys())
    # --- Force "divergence" (convergence) to be last ---
    if "divergence" in varNames:
        varNames = [v for v in varNames if v != "divergence"] + ["divergence"]
    n_vars = len(varNames)

    # --- Layout logic ---
    if len(modelLabels) == 2 and plottype == "TZ":
        n_cols = 3  # Model1, Model2, Difference
        n_rows = n_vars
        layout_mode = "comparison"
    else:
        n_cols = 3
        n_rows = int(np.ceil(n_vars / n_cols))
        layout_mode = "overlay"

    fig = plt.figure(figsize=(5.5 * n_cols, 3.5 * n_rows))
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, wspace=0.3, hspace=0.6)

    # --- Loop through variables ---
    
    for i, varName in enumerate(varNames):
        axisTitle = varName.replace("divergence","convergence")
        # ============================================================
        # TZ layout: 3 columns per variable (Model1, Model2, Δ)
        # ============================================================
        if layout_mode == "comparison":
            row = i

            # --- Compute shared clim for both models ---
            if varName not in ["refl10cm", "refl10cm_1km"]:
                out1 = combinedDict2[modelLabels[0]][varName]
                out2 = combinedDict2[modelLabels[1]][varName]
                vmin = np.nanmin([np.nanmin(out1), np.nanmin(out2)])
                vmax = np.nanmax([np.nanmax(out1), np.nanmax(out2)])
                if varName in ["w","divergence"]: #w
                    vlim = np.nanmax(np.abs([vmin, vmax]))
                    clim = (-vlim, vlim)
                else:
                    clim = (vmin, vmax)
            else:
                clim = None

            # --- Column 1: Model 1 ---
            ax1 = fig.add_subplot(gs[row, 0])
            PlotSingle(ax1, [combinedDict2[modelLabels[0]]], varName, time, pressure_profiles,
                       labels=[modelLabels[0]], plottype="TZ", clim=clim)
            ax1.set_title(f"{modelLabels[0]} {axisTitle}", fontsize=11)

            # --- Column 2: Model 2 ---
            ax2 = fig.add_subplot(gs[row, 1])
            PlotSingle(ax2, [combinedDict2[modelLabels[1]]], varName, time, pressure_profiles,
                       labels=[modelLabels[1]], plottype="TZ", clim=clim)
            ax2.set_title(f"{modelLabels[1]} {axisTitle}", fontsize=11)

            # --- Column 3: Difference ---
            ax3 = fig.add_subplot(gs[row, 2])
            PlotSingle(ax3, [combinedDict2[modelLabels[0]], combinedDict2[modelLabels[1]]],
                       varName, time, pressure_profiles,
                       labels=modelLabels, plottype="TZ")
            ax3.set_title(f"Δ({modelLabels[0]} - {modelLabels[1]}) {axisTitle}", fontsize=11)

        # ============================================================
        # T layout: overlay both models on same axes (line plots)
        # ============================================================
        elif layout_mode == "overlay":
            row, col = divmod(i, n_cols)
            ax = fig.add_subplot(gs[row, col])

            # define consistent colors for models
            model_colors = {"NSSL": "blue", "TEMPO": "green"}

            # plot both models on same axis
            for label in modelLabels:
                dataDictionary = combinedDict2[label]

                # track existing lines to only recolor new ones
                existing_lines = len(ax.get_lines())
                PlotSingle(ax, [dataDictionary], varName, time, pressure_profiles,
                           labels=[label], plottype="T")
                new_lines = ax.get_lines()[existing_lines:]

                for line in new_lines:
                    line.set_color(model_colors.get(label, "k"))
                    line.set_label(label)

            ax.legend(loc="upper left", fontsize=9)
            ax.set_title(axisTitle, fontsize=11)

    # ============================================================
    # Format axes and layout
    # ============================================================
    for ax in fig.get_axes():
        ax.tick_params(labelbottom=True)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    title_y_relative = AdjustLayout(fig)

    # global title
    label_text = " vs ".join(modelLabels)
    # plt.suptitle(f"{ModelData.region}_{ModelData.case} {label_text}",
    #              fontsize=16, fontweight="bold", y=title_y_relative)

    # ============================================================
    # Saving Figure
    # ============================================================
    # save figure
    label_text = label_text.replace(" ", "")
    return fig, combinedDict2, key, label_text


In [ ]:
####################################
#PLOTTING

In [ ]:
if plotting:
    # setting up dictionaries for plotting
    labels = ("NSSL", "TEMPO")
    
    # variable groups
    print('varGroups')
    varGroups = {
        "T_1": outputDictionary_2D_NSSL.keys()
    }
    
    # model source dictionaries
    print('modelDicts')
    modelDicts = {
        "NSSL": [outputDictionary_2D_NSSL],
        "TEMPO": [outputDictionary_2D_TEMPO],
    }
    
    # --- function ---
    def GetCombinedDictionary(varNames, dicts):
        """Get varName: value from the first dict in `dicts` that contains it."""
        #Format of combinedDictionary
        # Combined[key] = { "NSSL": {...}, "TEMPO": {...} }
        return {v: next((d[v] for d in dicts if v in d), None) for v in varNames}
    
    # --- build labeled combined structure ---
    print('combinedDictionary')
    combinedDictionary = {
        group: {
            label: GetCombinedDictionary(varNames, modelDicts[label])
            for label in labels
        }
        for group, varNames in varGroups.items()
    }

In [ ]:
if plotting:
    #2D Variable Plots
    [fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, key = "T_1", plottype="T")
    # SaveFigure(fig, combinedDict2, key, label_text)

In [ ]:
            # elif "qv" in varName:
            #     variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName="qv").isel(nVertLevels=0)
            #     if varName == "qv_prime":
            #         variableSubset -= variableSubset.mean()




In [ ]:
########################################
#TESTING

In [ ]:
# # def ComputeOutflowVelocity(u, v, th_rho):
# #     dthdx = th_rho.differentiate("x")
# #     dthdy = th_rho.differentiate("y")

# #     gradMag = np.sqrt(dthdx**2 + dthdy**2)

# #     nx = dthdx / gradMag
# #     ny = dthdy / gradMag

# #     Uout = u * nx + v * ny
# #     return Uout

# def ComputeOutflowVelocity(uField, vField, thetaRhoField):
#     """
#     Computes Eulerian outflow velocity on a latitude/longitude grid.
#     uField, vField must be in m/s.
#     thetaRhoField must be in K.

#     Physical gradient on a spherical lat/lon grid:

#         d(theta)/dx = (1 / (R * cos(phi))) * d(theta)/d(lambda)
#         d(theta)/dy = (1 / R) * d(theta)/d(phi)

#     where:
#         lambda = longitude (radians)
#         phi    = latitude  (radians)
#         R      = Earth radius

#     Outward unit vector:

#         nx = (d(theta)/dx) / |grad(theta)|
#         ny = (d(theta)/dy) / |grad(theta)|

#     Outflow velocity:

#         Uout = u * nx + v * ny
#     """

#     import numpy as np

#     # Earth's radius (meters)
#     R = 6_371_000  

#     latRad = np.deg2rad(thetaRhoField.latitude)

#     # --- compute correct physical gradients ---
#     dthdlat = thetaRhoField.differentiate("latitude")    # K per degree latitude
#     dthdlon = thetaRhoField.differentiate("longitude")   # K per degree longitude

#     # Convert degrees → meters
#     dthdy = dthdlat * (np.pi/180) * R
#     dthdx = dthdlon * (np.pi/180) * R * np.cos(latRad)

#     # --- gradient magnitude ---
#     gradMag = np.sqrt(dthdx**2 + dthdy**2)

#     # --- unit normal vector ---
#     nx = dthdx / gradMag
#     ny = dthdy / gradMag

#     # --- project wind onto outward direction ---
#     Uout = uField * nx + vField * ny
#     Uout.name = "outflow_velocity"

#     return Uout


# def Testing_ColdPools(ModelData, t=40):
    
#     th_rho = LoadAdditionalVariableData(ModelData, "Theta_rho", t=t)
#     th_rho_prime = th_rho - np.mean(th_rho)
    
#     qr = ModelData.GetDataTimestep(t=t,varName="qr",printout=False).isel(nVertLevels=0)
#     w = ModelData.GetDataTimestep(t=t,varName="w",printout=False).isel(nVertLevelsP1=0)
#     u = ModelData.GetDataTimestep(t=t,varName="uReconstructZonal",printout=False).isel(nVertLevels=0)
#     v = ModelData.GetDataTimestep(t=t,varName="uReconstructMeridional",printout=False).isel(nVertLevels=0)
#     VEL = np.sqrt(u**2+v**2)
#     OUTFLOW = ComputeOutflowVelocity(u,v,th_rho)
    
#     cond1 = th_rho_prime < -1
#     cond2 = qr > 1e-6
    
#     # fig, ax = plt.subplots()
#     # th_rho.plot(ax=ax)
#     # th_rho.where(cond1 & cond2).plot.contour(ax=ax,colors="white",linewidths=1.2,zorder=10)
#     return th_rho.where(cond1 & cond2), w.where(cond1 & cond2), VEL.where(cond1 & cond2), OUTFLOW.where(cond1 & cond2)


# #getting NSSL dictionaries
# RunType = (Region,Case,"NSSL",spinup_hours)
# ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
# #getting TEMPO dictionaries
# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# t=40
# [theta_rho1, w1, VEL1, OUTFLOW1] = Testing_ColdPools(ModelData_NSSL,t=t)
# [theta_rho2, w2, VEL2, OUTFLOW2] = Testing_ColdPools(ModelData_TEMPO,t=t)

# print(theta_rho1.mean().item()-theta_rho2.mean().item())
# print(w1.mean().item()-w2.mean().item())
# print(VEL1.mean().item()-VEL2.mean().item())
# print(OUTFLOW1.mean().item()-OUTFLOW2.mean().item())

In [ ]:
#testing gaussian filter for cps

# t=100
# th_rho = LoadAdditionalVariableData(ModelData, "Theta_rho", t=t)

# from scipy.ndimage import gaussian_filter
# th_rho_bg = xr.apply_ufunc(
#     gaussian_filter,
#     th_rho,
#     kwargs={"sigma": 3}
# )

# th_rho_prime1 = th_rho - th_rho.mean()
# th_rho_prime2 = th_rho - th_rho_bg
# th_rho_prime1.plot()
# th_rho_prime2.plot()
# th_rho_prime2.where(th_rho_prime2<-0.5).plot()

In [ ]:
#testing high qv at single time

# # ModelData.timeStrings[156]

# RunType = (Region,Case,"NSSL",spinup_hours)
# ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
# th_rho1 = LoadAdditionalVariableData(ModelData_NSSL, variableName="Theta_rho", t=t)
# th_rho_prime1 = th_rho1 - th_rho1.mean()

# th_rho2 = LoadAdditionalVariableData(ModelData_NSSL, variableName="Theta_rho", t=t)
# th_rho_prime2 = th_rho2 - th_rho2.mean()



# t=150

# data1 = ModelData_NSSL.GetDataTimestep(t=t)
# data2 = ModelData_TEMPO.GetDataTimestep(t=t)

# qr1 = data1['qr'].isel(nVertLevels=0)
# qr2 = data2['qr'].isel(nVertLevels=0)


# qv1 = data1['qv'].isel(nVertLevels=0).where((th_rho_prime1<-1) & (qr1>1e-6))*1e3
# qv_prime1 = qv1 - qv1.mean()

# qv2 = data2['qv'].isel(nVertLevels=0).where((th_rho_prime2<-1) & (qr2>1e-6))*1e3
# qv_prime2 = qv2 - qv2.mean()

In [ ]:
# #testing high qv at some times

# qvList1 = []
# qvList2 = []

# for t in range(150, 161):
#     data1 = ModelData_NSSL.GetDataTimestep(t=t)
#     data2 = ModelData_TEMPO.GetDataTimestep(t=t)

#     thRho1 = LoadAdditionalVariableData(ModelData_NSSL, variableName="Theta_rho", t=t)
#     thRhoPrime1 = thRho1 - thRho1.mean()

#     thRho2 = LoadAdditionalVariableData(ModelData_NSSL, variableName="Theta_rho", t=t)
#     thRhoPrime2 = thRho2 - thRho2.mean()

#     qr1 = data1['qr'].isel(nVertLevels=0)
#     qr2 = data2['qr'].isel(nVertLevels=0)

#     qv1 = data1['qv'].isel(nVertLevels=0).where((thRhoPrime1 < -1) & (qr1 > 1e-6)) * 1e3
#     qvPrime1 = qv1 - qv1.mean()

#     qv2 = data2['qv'].isel(nVertLevels=0).where((thRhoPrime2 < -1) & (qr2 > 1e-6)) * 1e3
#     qvPrime2 = qv2 - qv2.mean()

#     qvList1.append(qv1.mean().item())
#     qvList2.append(qv2.mean().item())



#     # # Plot for qv1 at this timestep
#     # plt.figure()
#     # qvPrime1.plot()
#     # plt.title(f"qvPrime1 at t = {t}")
#     # plt.show()

#     # # Plot for qv2 at this timestep
#     # plt.figure()
#     # qvPrime2.plot()
#     # plt.title(f"qvPrime2 at t = {t}")
#     # plt.show()


# plt.plot(qvList1)
# plt.plot(qvList2)